# University Campus Routing Problem 

In [137]:
import heapq # We use this module because it implements priority queue
import math # For Euclidian distance
import itertools

### Define classes that the system will use 

In [138]:
class Node: 
    def __init__(self, coordenate, parent = None, path_cost = 0, floor = 1, building = "...", action = None):
        self.coordenate = coordenate
        self.parent = parent 
        self.path_cost = path_cost
        self.floor = floor
        self.building = building
        self.action = action
        # We inicialize the node for root

class Problem: 
    def __init__(self, start, goal):
        self.start = start
        self.goal = goal 
        

### Moves availaible on the problem while Walking

In [139]:
movesWhenWalking = {
    'Down': (0, 1),
    'Up': (0, -1),
    'Right': (1, 0),
    'Left': (-1, 0)
}

### Define actions availaible when walking


In [140]:
def actionsWhenWalking(node, grid):
    x, y = node.coordenate
    available_actions = {}

    for direction, (dx, dy) in movesWhenWalking.items():
        newX = x + dx
        newY = y + dy

        # We check if the new position is on the map
        isInsideMap = (0 <= newY < len(grid)) and (0 <= newX < len(grid[0]))

        if isInsideMap:
            if grid[newY][newX] != "#":
                available_actions[direction] = True 
            else:
                available_actions[direction] = False
        else:
            available_actions[direction] = False 

    return available_actions


### Define actions availaible when inside a building

In [141]:
buildingWithElevator = {
    'B32', 
    'B33',
    'B35',
    'B37', 
    'B38'
}

def actionsWhenBuilding(node): 
    available_actions = []
    
    
    if node.floor > 1:
        available_actions.append("bajar_escaleras")
    if node.floor < 3: 
        available_actions.append("subir_escaleras")


    if node.building in buildingWithElevator:
        available_actions.append("subir_ascensor")
        available_actions.append("bajar_ascensor")
        

    return available_actions


### Path Cost for building

In [142]:
def path_cost_for_building(node, action): 
    if action in ("subir_ascensor", "bajar_ascensor"):
        if node.action in ("subir_ascensor", "bajar_ascensor"):
            return 0
        else:
            return 3    
    else:
        return 1     # Each other action (going down/up stairs) as cost 1 


def result_when_building(node, action):
    if action == "subir_escaleras" or action == "subir_ascensor":
        new_floor = node.floor + 1
    else:
        new_floor = node.floor - 1

    return Node(
        coordenate=node.coordenate,   # This because we still in the same position in (x,y)
        parent=node,
        path_cost=node.path_cost + path_cost_for_building(node, action),
        floor=new_floor,
        building=node.building,
        action=action
    )


### Best First Search g(n) for path cost 

In [143]:
def best_first_search_building(start_node, problem):
    frontier = []
    counter = itertools.count()
    
    heapq.heappush(frontier, (start_node.path_cost, next(counter), start_node))
    reached = {(start_node.floor, start_node.action): start_node.path_cost}

    while frontier:
        cost, _, node = heapq.heappop(frontier)

        if is_full_goal(problem, node):
            return node

        for action in actionsWhenBuilding(node):
            child = result_when_building(node, action)
            g = child.path_cost

            state_id = (child.floor, child.action)

            if state_id not in reached or g < reached[state_id]:
                reached[state_id] = g
                heapq.heappush(frontier, (g, next(counter), child))

    return None


### Related with Heuristic Function 

In [144]:
def euclidean_distance(a, b):
    
    return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)

def heuristicFunction(problem, node): 
    return euclidean_distance(node.coordenate,problem.goal["coordenate"])

### Goal State

In [145]:
def is_partial_goal(problem, node): 
    if heuristicFunction(problem, node) == 0: 
        return True 
    return False

def is_full_goal(problem, node): 
    if node.floor == problem.goal["floor"]: 
        return True 
    return False 

### Expand Nodes

In [146]:
def expand(problem, node, grid):
    
    available_actions = actionsWhenWalking(node, grid)

    for direction, can_move in available_actions.items():
        if can_move:
            x, y = node.coordenate
            dx, dy = movesWhenWalking[direction]
            new_coord = (x + dx, y + dy)

            cell = grid[new_coord[1]][new_coord[0]]

            
            if isinstance(cell, str) and cell.startswith("B"):
                new_building = cell
            else:
                new_building = node.building

            new_node = Node(
                coordenate=new_coord,
                parent=node,
                path_cost=node.path_cost + 1,
                floor=node.floor,
                building=new_building,
                action=direction
            )

            yield new_node

### A* search. 
In this case we mixed Breadth-First-Search with Greedy Search. Due to each plane movement has 1 unit value.

In [147]:
def a_star(problem, grid):
    start_node = Node(coordenate=problem.start["coordenate"], floor=problem.start["floor"], building=problem.start["edificio"])
    # if start_node.floor != 1:
    #     goal_floor_1 = {"floor": 1}
    #     temp_problem = Problem(start=problem.start, goal=goal_floor_1)
    #     start_node = best_first_search_building(start_node, temp_problem)
    #     if start_node is None:
    #         return None
    frontier = []
    counter = itertools.count()  
    heapq.heappush(frontier, (heuristicFunction(problem, start_node), next(counter), start_node))
    reached = {start_node.coordenate: 0}
    
    while frontier:
        _, _, node = heapq.heappop(frontier)

        #if (node.coordenate != problem.goal["coordenate"] and node.floor != 1): 
        #    node = best_first_search_building(node,problem, False)
        if node.coordenate != problem.goal["coordenate"] and node.floor != 1: 
            goal_floor_1 = {"floor": 1}
            temp_problem = Problem(start=problem.start, goal=goal_floor_1)
            node = best_first_search_building(node, temp_problem)

        if node.coordenate == problem.goal["coordenate"]:
            
            node = best_first_search_building(node, problem)
            return node
        
        for child in expand(problem, node, grid):
            g = child.path_cost
            if child.coordenate not in reached or g < reached[child.coordenate]:
                reached[child.coordenate] = g
                f = g + heuristicFunction(problem, child)
                heapq.heappush(frontier, (f, next(counter), child))
    
    return None


### Reconstruct Path

In [148]:
def reconstruct_path(node):
    
    path = []
    states = []
    current = node
    
    while current.parent is not None:  # Go until initial state
        path.append(current.action)
        states.append(current)
        current = current.parent

    states.append(current)  
    path.reverse()          
    states.reverse()        
    
    return path, states


### Define problem 

In [149]:
problem = Problem(
    start={"coordenate": (2,2), "floor":3, "edificio":"B30"},
    goal={"coordenate": (16,6), "floor":3, "edificio":"B38"},
)

grid = [
    [" ", "#", " ",  " ", "#",  " ",  " ", "#", " ",  " ", "#", " ",  " ", "#", " ",  " ", " "],
    [" ", "#", " ",  " ", "#",  " ",  " ", "#", " ",  " ", "#", " ",  " ", "#", " ",  " ", " "],
    [" ", " ", "B30"," ", " ", "B31"," ", " ", "B32"," ", " ", "B33"," ", " ", "B34"," ", " "],
    [" ", "#", " ",  "#", "#", " ",  "#", " ", "#",  "#", " ", "#",  " ", "#", " ",  "#", " "],
    [" ", "#", " ",  " ", " ", " ",  "#", " ", "#",  " ", " ", "#",  " ", "#", " ",  "#", " "],
    [" ", " ", " ",  "#", " ", " ",  " ", " ", " ",  "#", " ", " ",  " ", " ", " ",  "#", " "],
    [" ", "#", " ",  " ", "B35"," ",  "#", " ", "B36"," ", "#", " ",  "B37"," ", "#", " ", "B38"],
    [" ", "#", "#",  " ", "#", " ",  "#", " ", "#",  " ", "#", " ",  "#", " ", "#",  " ", " "],
    [" ", " ", " ",  " ", "#", " ",  " ", " ", "#",  " ", " ", " ",  "#", " ", " ",  " ", " "],
    ["#", "#", " ",  "#", "#", " ",  "#", " ", " ",  " ", "#", " ",  "#", " ", "#",  "#", " "],
    [" ", " ", " ",  " ", "#", " ",  "C", " ", "#",  " ", "#", " ",  " ", " ", "B",  " ", " "],
    [" ", "#", "#",  " ", "#", " ",  "#", " ", "#",  " ", "#", " ",  "#", " ", "#",  " ", " "],
    [" ", " ", " ",  " ", " ", " ",  " ", " ", " ",  " ", " ", " ",  " ", " ", " ",  " ", " "],
    [" ", "#", " ",  "#", " ", "#",  " ", "#", " ",  "#", " ", "#",  " ", "#", " ",  "#", " "],
    [" ", "#", " ",  " ", " ", " ",  " ", " ", " ",  " ", " ", " ",  " ", " ", " ",  "#", " "],
    [" ", " ", " ",  "#", "#", "#",  " ", "#", "#",  "#", " ", "#",  "#", "#", " ",  " ", " "],
]

### Solve the problem 

In [150]:
result_node = a_star(problem, grid)

# Reconstruct path and states
ruta, estados = reconstruct_path(result_node)

print("Path:", ruta)
print("Total cost:", result_node.path_cost)
print("Position:", [(s.coordenate, s.floor) for s in estados])


Path: ['bajar_escaleras', 'bajar_escaleras', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Down', 'Down', 'Down', 'Down', 'subir_escaleras', 'subir_escaleras']
Total cost: 22
Position: [((2, 2), 3), ((2, 2), 2), ((2, 2), 1), ((3, 2), 1), ((4, 2), 1), ((5, 2), 1), ((6, 2), 1), ((7, 2), 1), ((8, 2), 1), ((9, 2), 1), ((10, 2), 1), ((11, 2), 1), ((12, 2), 1), ((13, 2), 1), ((14, 2), 1), ((15, 2), 1), ((16, 2), 1), ((16, 3), 1), ((16, 4), 1), ((16, 5), 1), ((16, 6), 1), ((16, 6), 2), ((16, 6), 3)]
